# EMA + RSI

EMA Crossover + RSI Filter \
It uses ATR(14) for dynamic stops/tolerance (adapts to volatility). \
Momentum strategy, excellent for scalping and intraday across all TFs. \
Filters false signals with RSI.

__How EMA+RSI Algorithm Determines Entry/Exit:__
- Fast EMA (9) / Slow EMA (21) – standard for crypto.
- Long Entry: Fast EMA crosses above Slow EMA AND RSI(14) < 70 (not overbought).
- Short Entry: Fast EMA crosses below Slow EMA AND RSI(14) > 30 (not oversold).
- Exit: Reverse crossover OR price hits ATR-based trailing stop.
- Excellent filter reduces whipsaws in ranging markets.

## Configuration

In [1]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [2]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import StrategyConfig
from engine.visualization import build_chart

In [ ]:
# Dataset is configured globally in engine/data_configurator.py (the ACTIVE spec).
# Edit ACTIVE there to change symbol / interval / window for every notebook at once.
from engine.data_configurator import ACTIVE, save_result

SYMBOL, INTERVAL = ACTIVE.symbol, ACTIVE.interval

In [ ]:
# Candles come from the shared cache via the global ACTIVE spec.
from engine.data_configurator import load_data

df = load_data()

## EMA + RSI

In [5]:
# Import EMA + RSI strategy
from engine.strategies import EMACrossoverStrategy

In [ ]:
# Backtest EMA + RSI strategy
config = StrategyConfig()
strategy = EMACrossoverStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# EMA + RSI strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()